# 03 Models: Logistic Regression, Naive Bayes, Decision Tree

Compares the three models trained with `src/train.py`. Each one was tuned on the 1M sample using val macro-F1, then the best setting was refit on all of train and scored once on val and test.

To make the results files again:

```
python -m src.train --model lr
python -m src.train --model nb
python -m src.train --model dt
python -m src.train --model dt --extra year Source
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))  # so src imports from the notebooks folder
from src import config
from src.models import fit_model, make_model

config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# same style as notebooks 01 and 02
INK = '#52514e'
GRAY = '#8a8980'
plt.rcParams.update({
    'figure.facecolor': '#fcfcfb', 'axes.facecolor': '#fcfcfb',
    'axes.edgecolor': '#c3c2b7', 'axes.labelcolor': INK,
    'xtick.color': INK, 'ytick.color': INK, 'text.color': '#1a1a19',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#e1e0d9', 'grid.linewidth': 0.8,
    'axes.axisbelow': True, 'font.size': 10,
})

# one color per model, same everywhere
MODELS = {
    'lr': ('Logistic Regression', '#2a78d6'),
    'nb': ('Naive Bayes', '#eb6834'),
    'dt': ('Decision Tree', '#1baf7a'),
    'dt_plus_year_source': ('Decision Tree + year/Source', '#eda100'),
}
MAIN = ['lr', 'nb', 'dt']

## Results

The baseline always guesses severity 2, the most common class.

In [ ]:
def load(name):
    with open(config.RESULTS_DIR / f'{name}.json') as f:
        return json.load(f)

results = {m: {split: load(f'{m}_{split}') for split in ['val', 'test']} for m in MODELS}

# baseline from the test class counts: always guess 2
support = {c: results['dt']['test']['per_class'][str(c)]['support'] for c in config.CLASSES}
share_2 = support[2] / sum(support.values())
baseline = {'accuracy': share_2, 'macro_f1': (2 * share_2 / (1 + share_2)) / 4}

rows = []
for m, (label, _) in MODELS.items():
    rows.append({
        'model': label,
        'val macro-F1': results[m]['val']['macro_f1'],
        'test macro-F1': results[m]['test']['macro_f1'],
        'test accuracy': results[m]['test']['accuracy'],
    })
rows.append({'model': 'Always guess 2', 'val macro-F1': np.nan,
             'test macro-F1': baseline['macro_f1'], 'test accuracy': baseline['accuracy']})
summary = pd.DataFrame(rows).set_index('model')
summary.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
names = list(MODELS)
scores = [results[m]['test']['macro_f1'] for m in names]
ax.barh([MODELS[m][0] for m in names], scores, color=[MODELS[m][1] for m in names], height=0.6)
ax.axvline(baseline['macro_f1'], color=GRAY, linestyle='--', linewidth=1.5)
ax.text(baseline['macro_f1'] + 0.005, 0.5, 'always guess 2', va='center', color=INK, fontsize=9)
for i, s in enumerate(scores):
    ax.text(s + 0.005, i, f'{s:.3f}', va='center', color=INK)
ax.invert_yaxis()
ax.set_xlim(0, 0.6)
ax.set_xlabel('test macro-F1')
ax.set_title('Test macro-F1 by model')
ax.grid(axis='y', visible=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'model_macro_f1.png', dpi=150, bbox_inches='tight')
plt.show()

## F1 for each severity

Macro-F1 is the average of these four, so a model only scores well if it finds the rare severities too.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.2
x = np.arange(len(config.CLASSES))
for i, m in enumerate(MODELS):
    f1 = [results[m]['test']['per_class'][str(c)]['f1'] for c in config.CLASSES]
    ax.bar(x + (i - 1.5) * width, f1, width - 0.02, color=MODELS[m][1], label=MODELS[m][0])
ax.set_xticks(x, [f'Severity {c}' for c in config.CLASSES])
ax.set_ylabel('test F1')
ax.set_ylim(0, 1)
ax.set_title('Test F1 by severity')
ax.legend(frameon=False, fontsize=9)
ax.grid(axis='x', visible=False)
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'model_f1_by_severity.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion matrices

Each row is the true severity, split by what the model guessed (row %). A good model has dark squares on the diagonal.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, m in zip(axes, MAIN):
    cm = np.array(results[m]['test']['confusion_matrix'])
    pct = cm / cm.sum(axis=1, keepdims=True) * 100
    ax.imshow(pct, cmap='Blues', vmin=0, vmax=100)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f'{pct[i, j]:.0f}', ha='center', va='center',
                    color='white' if pct[i, j] > 50 else '#1a1a19')
    ax.set_xticks(range(4), config.CLASSES)
    ax.set_yticks(range(4), config.CLASSES)
    ax.set_xlabel('predicted')
    ax.set_title(MODELS[m][0])
    ax.grid(False)
axes[0].set_ylabel('true')
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / 'model_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

## Twin check

The split is random, so some test rows have the exact same feature values as a row in train (same time, same weather report, same road). A deep tree can just remember those. This scores the best tree and naive bayes again, split into test rows with and without a twin in train.

This refits both models on all of train, so it takes a few minutes.

In [ ]:
features = config.MODEL_FEATURES
train = pd.read_parquet(config.TRAIN_PARQUET, columns=features + [config.TARGET])
test = pd.read_parquet(config.TEST_PARQUET, columns=features + [config.TARGET])

# same features as a row in train
def row_key(df):
    return pd.util.hash_pandas_object(df[features].astype(str), index=False)

twin = row_key(test).isin(set(row_key(train))).values
print(f'test rows with a twin in train: {twin.mean():.1%}')

In [ ]:
from sklearn.metrics import f1_score

best = {
    'nb': make_model('nb', features, model__alpha=0.01),
    'dt': make_model('dt', features, model__max_depth=None, model__min_samples_leaf=1),
}
y = test[config.TARGET].values
twin_rows = []
for m, pipe in best.items():
    pred = fit_model(pipe, train[features], train[config.TARGET]).predict(test[features])
    for group, mask in [('has twin', twin), ('no twin', ~twin)]:
        twin_rows.append({'model': MODELS[m][0], 'test rows': group,
                          'macro-F1': f1_score(y[mask], pred[mask], labels=config.CLASSES,
                                               average='macro', zero_division=0)})
twin_table = pd.DataFrame(twin_rows).pivot(index='model', columns='test rows', values='macro-F1')
twin_table.round(3)

## Observations

- Decision tree is the best of the three (test macro-F1 0.414), then naive bayes (0.328), then logistic regression (0.238). All three beat always guessing 2 (0.220).
- None of them beat the 0.787 accuracy you get from always guessing 2, so accuracy isn't a good way to compare models here. Macro-F1 is.
- Logistic regression barely beats the baseline. It treats hour, month and weekday as straight-line numbers, so it can't pick up things like rush hour peaks.
- GaussianNB scored below the baseline because most features are 0/1, so we switched to CategoricalNB with binned weather numbers.
- Severity 4 is the hardest. Only the decision tree finds some of it (F1 0.24), naive bayes almost never predicts it.
- Part of the tree's score is memorizing. 28% of test rows have the same features as a train row, and the tree scores 0.601 on those but 0.343 on the rest. Naive bayes is about the same on both (0.339 vs 0.322), so the real gap between them is small.
- Adding year and Source pushes the tree to 0.505, but that's from how each source labels severity, not the crash itself. Severity 4 actually gets worse (0.24 to 0.12), so they stay out.

## Additional models: KNN, Random Forest, and XGBoost

The saved results of the models use different training scopes, which are noted below:

- KNN uses 100,000 rows each for training, validation, and test. This prevents excessive run time and memory hiccups due to tuning.
- Random Forest was tuned on a 1M-row training sample. A full train refit exceeded available memory, so the model was refit on that 1M-row sample and evaluated on the full validation and test splits.
- XGBoost was tuned on a 1M-row sample, then refit on all training rows and evaluated on the full validation and test splits.

Commands used:

```
python -m src.train --model knn --quick --quick-rows 100000
python -m src.train --model rf --refit-sample
python -m src.train --model xgb

In [ ]:

def load_optional(name, result_dir):
    path = result_dir / f"{name}.json"
    if not path.exists():
        return None
    with path.open() as f:
        return json.load(f)


ADDITIONAL_MODELS = {
    "knn": ("KNN (100K quick)", "#756bb1"),
    "rf": ("Random Forest (1M refit)", "#187c9c"),
    "xgb": ("XGBoost (full refit)", "#d14b78"),
}

RESULT_FILES = {
    "knn": ("knn_val", "knn_test", config.RESULTS_DIR / "quick"),
    "rf": ("rf_sampled_val", "rf_sampled_test", config.RESULTS_DIR),
    "xgb": ("xgb_val", "xgb_test", config.RESULTS_DIR),
}

additional_results = {}
missing_results = []

for key in ADDITIONAL_MODELS:
    val_name, test_name, result_dir = RESULT_FILES[key]
    val_result = load_optional(val_name, result_dir)
    test_result = load_optional(test_name, result_dir)

    if val_result is None or test_result is None:
        missing_results.append(key)
    else:
        additional_results[key] = {"val": val_result, "test": test_result}

print("Missing result files:", ", ".join(missing_results) if missing_results else "none")

rows = []
for key, (label, _) in ADDITIONAL_MODELS.items():
    if key in additional_results:
        result = additional_results[key]
        rows.append({
            "model": label,
            "validation macro-F1": result["val"]["macro_f1"],
            "test macro-F1": result["test"]["macro_f1"],
            "test accuracy": result["test"]["accuracy"],
        })

additional_summary = (
    pd.DataFrame(rows).set_index("model")
    if rows
    else pd.DataFrame(
        columns=["validation macro-F1", "test macro-F1", "test accuracy"]
    )
)
additional_summary.round(3)


### Test macro-F1 comparison for KNN, RF, XGB

In [ ]:
if additional_summary.empty:
    print("Run knn, rf, and xgb training first to populate this plot.")
else:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    labels = additional_summary.index.tolist()
    scores = additional_summary["test macro-F1"].tolist()
    colors = [
        ADDITIONAL_MODELS[key][1]
        for key in ADDITIONAL_MODELS
        if key in additional_results
    ]
    ax.axvline(baseline['macro_f1'], color=GRAY, linestyle='--', linewidth=1.5)
    ax.text(baseline['macro_f1'] + 0.005, 0.5, 'always guess 2', va='center', color=INK, fontsize=9)
    ax.barh(labels, scores, color=colors, height=0.6)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.6, max(scores) + 0.05))
    ax.set_xlabel("test macro-F1")
    ax.set_title("Additional models: test macro-F1")
    ax.grid(axis="y", visible=False)

    for i, score in enumerate(scores):
        ax.text(score + 0.005, i, f"{score:.3f}", va="center", color=INK)

    fig.tight_layout()
    fig.savefig(
        config.FIGURES_DIR / "additional_model_macro_f1.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()


### Test F1 By Severity for KNN, RF, XGB

In [ ]:
model_f1_rows = []
model_colors = {}

for key, (label, color) in ADDITIONAL_MODELS.items():
    if key not in additional_results:
        continue

    model_colors[label] = color
    per_class = additional_results[key]["test"]["per_class"]

    for severity in config.CLASSES:
        model_f1_rows.append({
            "model": label,
            "severity": severity,
            "test F1": per_class[str(severity)]["f1"],
        })

model_f1_table = (
    pd.DataFrame(model_f1_rows)
    .pivot(index="severity", columns="model", values="test F1")
    .reindex(config.CLASSES)
)

ax = model_f1_table.plot(
    kind="bar",
    figsize=(9, 5),
    color=[model_colors[name] for name in model_f1_table.columns],
    rot=0,
)
ax.set_title("Test F1 by Severity: KNN, Random Forest, and XGBoost")
ax.set_xlabel("Severity")
ax.set_ylabel("Test F1")
ax.set_ylim(0, 1)
ax.legend(title="Model")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

ax.figure.savefig(
    config.FIGURES_DIR / "knn_rf_xgb_f1_by_severity.png",
    dpi=300,
    bbox_inches="tight",
)

### Random Forest feature-set experiment

Run:

```
python -m src.train --model rf --feature-set weather --refit-sample
python -m src.train --model rf --feature-set time --refit-sample
python -m src.train --model rf --feature-set road --refit-sample
```


In [ ]:
FEATURE_SETS = {
    "weather": "Weather only",
    "time": "Time only",
    "road": "Road only",
}

feature_rows = []
missing_feature_sets = []
for feature_set, label in FEATURE_SETS.items():
    val = load_optional(f"rf_only_{feature_set}_sampled_val")
    test = load_optional(f"rf_only_{feature_set}_sampled_test")
    if val is None or test is None:
        missing_feature_sets.append(feature_set)
        continue
    feature_rows.append({
        "feature group": label,
        "validation macro-F1": val["macro_f1"],
        "test macro-F1": test["macro_f1"],
        "test accuracy": test["accuracy"],
    })

print(
    "Missing feature-set result files:",
    ", ".join(missing_feature_sets) if missing_feature_sets else "none",
)

rf_feature_summary = (
    pd.DataFrame(feature_rows).set_index("feature group")
    if feature_rows
    else pd.DataFrame(
        columns=["validation macro-F1", "test macro-F1", "test accuracy"]
    )
)
rf_feature_summary.round(3)


### Random Forest test macro-F1 by feature group

In [ ]:
if rf_feature_summary.empty:
    print("Run the three RF feature-set commands to populate this comparison.")
else:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    labels = rf_feature_summary.index.tolist()
    scores = rf_feature_summary["test macro-F1"].tolist()
    colors = ["#187c9c", "#5aa68f", "#eda100"][:len(labels)]
    ax.axvline(baseline['macro_f1'], color=GRAY, linestyle='--', linewidth=1.5)
    ax.text(baseline['macro_f1'] + 0.005, 0.5, 'always guess 2', va='center', color=INK, fontsize=9)
    ax.barh(labels, scores, color=colors, height=0.6)
    ax.invert_yaxis()
    ax.set_xlim(0, max(0.6, max(scores) + 0.05))
    ax.set_xlabel("test macro-F1")
    ax.set_title("Random Forest: test macro-F1 by feature group")
    ax.grid(axis="y", visible=False)

    for i, score in enumerate(scores):
        ax.text(score + 0.005, i, f"{score:.3f}", va="center", color=INK)

    fig.tight_layout()
    fig.savefig(
        config.FIGURES_DIR / "rf_feature_set_macro_f1.png",
        dpi=150,
        bbox_inches="tight",
    )
    plt.show()


### Random Forest class-imbalance experiment

In [ ]:
from sklearn.metrics import accuracy_score, recall_score

imbalance_train = pd.read_parquet(config.TRAIN_SAMPLE_PARQUET)
imbalance_val = pd.read_parquet(config.VAL_PARQUET)
imbalance_test = pd.read_parquet(config.TEST_PARQUET)
imbalance_features = config.FEATURE_SETS["combined"]

# fixed settings so only class weighting changes
rf_params = {
    "model__n_estimators": 300,
    "model__max_depth": None,
    "model__min_samples_leaf": 1,
}

imbalance_rows = []
class_rows = []

for use_weights in (False, True):
    label = "Class-weighted" if use_weights else "Unweighted"
    model = make_model("rf", imbalance_features, **rf_params)
    fit_model(
        model,
        imbalance_train[imbalance_features],
        imbalance_train[config.TARGET],
        balanced=use_weights,
    )

    val_pred = model.predict(imbalance_val[imbalance_features])
    test_pred = model.predict(imbalance_test[imbalance_features])
    y_val = imbalance_val[config.TARGET]
    y_test = imbalance_test[config.TARGET]

    imbalance_rows.append({
        "model": label,
        "validation macro-F1": f1_score(
            y_val, val_pred, labels=config.CLASSES,
            average="macro", zero_division=0,
        ),
        "test macro-F1": f1_score(
            y_test, test_pred, labels=config.CLASSES,
            average="macro", zero_division=0,
        ),
        "test accuracy": accuracy_score(y_test, test_pred),
    })

    class_f1 = f1_score(
        y_test, test_pred, labels=config.CLASSES,
        average=None, zero_division=0,
    )
    class_recall = recall_score(
        y_test, test_pred, labels=config.CLASSES,
        average=None, zero_division=0,
    )
    for severity, f1, recall in zip(config.CLASSES, class_f1, class_recall):
        class_rows.append({
            "model": label,
            "severity": severity,
            "test F1": f1,
            "test recall": recall,
        })

rf_imbalance_summary = pd.DataFrame(imbalance_rows).set_index("model").round(3)
rf_imbalance_class_summary = (
    pd.DataFrame(class_rows)
    .pivot(index="severity", columns="model", values=["test F1", "test recall"])
    .round(3)
)

rf_imbalance_summary, rf_imbalance_class_summary

### RF Test F1 class-imbalance Comparison by Severity

In [ ]:
import matplotlib.pyplot as plt

imbalance_f1 = rf_imbalance_class_summary["test F1"].reindex(
    columns=["Unweighted", "Class-weighted"]
)

ax = imbalance_f1.plot(kind="bar", figsize=(8, 4), rot=0)
ax.set_title("RF Test F1 by Severity: Class-Weighting Comparison")
ax.set_xlabel("Severity")
ax.set_ylabel("Test F1")
ax.set_ylim(0, 1)
ax.legend(title="RF variant")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

ax.figure.savefig(
    config.FIGURES_DIR / "rf_class_imbalance_f1.png",
    dpi=300,
    bbox_inches="tight",
)

## Observations

- Overall, all three models have test macro-F1 values higher than just guessing 2 (0.220). The Random Forest had the highest value of the three with 0.394, then comes XGBoost at 0.318, and last is KNN at 0.300.
- It should be noted of the possibility that KNN may have the lowest test macro-F1 result due to the smaller sample size (100k). However, XGBoost used the full train data and did not exceed the result of the RF, which only used 1M rows. The three models used different training sizes, and comparison is not perfectly controlled.
- While KNN matches RF in its test F1 values for predicting Severity 2, in all other severities, RF outperforms KNN.
- XGB matches RF in its test F1 values for predicting Severity 3 and 4, but in predicting Severities 1 and 2, RF outperforms XGB.
- For feature-set experiments on RF, both time and road only practically matched the test macro-F1 value of just guessing 2, with 0.220 and 0.222 respectively. Using weather only had a result of 0.355, which is the highest of the three, but still did not exceed the result of Random Forest with all features used (0.394).
- Class-imbalance testing on RF shows that class-weighting increased the test F1 results of the under-represented Severity 1, 3, and 4, but slightly decreased the prediction accuracy of guessing Severity 2. Class weighting also increased test macro-F1 from 0.325 to 0.397.